# Double-delta filter through the production builder

This notebook builds `DoubleDeltaFilter` through `UniPolfed.build_filter`; it does not preserve hand-written coefficients. The combined polynomial maps the left target to `+1` and the right target to `-1`, while the requested windows remain inside their resolved passbands.

In [ ]:
import Pkg
const NOTEBOOK_ROOT = normpath(joinpath(@__DIR__, ".."))
Pkg.activate(NOTEBOOK_ROOT)

using UniPolfed
const Interval = UniPolfed.Interval
const build_filter = UniPolfed.build_filter
scalar_polynomial(filter) = UniPolfed._scalar_filter(filter)
const RUN_PLOTS = false


## Build one asymmetric pair

The builder jointly resolves the two Chebyshev orders, repairs interval coverage, normalizes the two peaks, and resolves the effective passbands.

In [ ]:
interval1 = Interval(-0.45, -0.35)
interval2 = Interval(0.15, 0.30)
filter = build_filter(
    UniPolfed.DoubleDeltaFilter, interval1, interval2;
    cutoff_lower=-0.25,
    cutoff_upper=0.25,
    order_safety_factor=0.97,
)
P = scalar_polynomial(filter)

println("orders: K1=$(filter.order1), K2=$(filter.order2)")
println("targets: $(filter.target1) → $(P(filter.target1)), $(filter.target2) → $(P(filter.target2))")
println("passband 1: [$(filter.passband1.left), $(filter.passband1.right)]")
println("passband 2: [$(filter.passband2.left), $(filter.passband2.right)]")


## Verify the filter contract

These checks mirror the important production invariants: ordered windows, interval coverage, cutoff crossings, normalized peaks, and the package leakage validator.

In [ ]:
function verify_double_delta(filter; atol=1e-8)
    P = scalar_polynomial(filter)
    checks = [
        "windows are ordered" => filter.interval1.right < filter.interval2.left,
        "passbands are ordered" => filter.passband1.right < filter.passband2.left,
        "passband 1 covers interval 1" =>
            filter.passband1.left <= filter.interval1.left && filter.passband1.right >= filter.interval1.right,
        "passband 2 covers interval 2" =>
            filter.passband2.left <= filter.interval2.left && filter.passband2.right >= filter.interval2.right,
        "left peak is +1" => isapprox(P(filter.target1), 1.0; atol),
        "right peak is -1" => isapprox(P(filter.target2), -1.0; atol),
        "passband 1 meets upper cutoff" =>
            isapprox(P(filter.passband1.left), filter.cutoff_upper; atol) &&
            isapprox(P(filter.passband1.right), filter.cutoff_upper; atol),
        "passband 2 meets lower cutoff" =>
            isapprox(P(filter.passband2.left), filter.cutoff_lower; atol) &&
            isapprox(P(filter.passband2.right), filter.cutoff_lower; atol),
        "production leakage validation" => UniPolfed.validate_filter(filter) === true,
    ]
    foreach(check -> println(check.second ? "✓ " : "✗ ", check.first), checks)
    @assert all(last, checks)
    return true
end

verify_double_delta(filter)


## Small configuration sweep

Rebuilding is cheap enough to inspect symmetric, asymmetric, close, and near-edge window pairs. The sweep records resolved orders instead of carrying old generated plots.

In [ ]:
cases = [
    (Interval(-0.30, -0.20), Interval(0.20, 0.30)),
    (Interval(-0.50, -0.40), Interval(0.10, 0.30)),
    (Interval(-0.15, -0.05), Interval(0.05, 0.15)),
    (Interval(-0.75, -0.65), Interval(0.10, 0.20)),
]

function first_valid_double(left, right)
    for cutoff in 0.17:0.01:0.35
        candidate = build_filter(
            UniPolfed.DoubleDeltaFilter, left, right;
            cutoff_lower=-cutoff, cutoff_upper=cutoff,
        )
        UniPolfed.validate_filter(candidate) && return candidate
    end
    error("double-delta cutoff ladder was exhausted")
end

sweep = map(cases) do (left, right)
    candidate = first_valid_double(left, right)
    (; left, right, K1=candidate.order1, K2=candidate.order2,
       total_order=UniPolfed.polynomial_order(candidate))
end
sweep


## Optional visualization

Set `RUN_PLOTS=true` above to plot the built polynomial, requested intervals, passbands, and cutoff levels. Plotting is optional so the numerical contract remains a quick smoke check.

In [ ]:
if RUN_PLOTS
    using PythonPlot
    x = range(-1.0, 1.0; length=2_000)
    figure(figsize=(10, 4))
    plot(x, P.(x); color="black", label="P(x)")
    axhline(filter.cutoff_upper; color="tab:blue", linestyle="--", label="upper cutoff")
    axhline(filter.cutoff_lower; color="tab:orange", linestyle="--", label="lower cutoff")
    axvspan(filter.interval1.left, filter.interval1.right; color="tab:blue", alpha=0.20)
    axvspan(filter.interval2.left, filter.interval2.right; color="tab:orange", alpha=0.20)
    axvspan(filter.passband1.left, filter.passband1.right; color="tab:blue", alpha=0.07)
    axvspan(filter.passband2.left, filter.passband2.right; color="tab:orange", alpha=0.07)
    xlabel("rescaled energy"); ylabel("P(x)"); legend(); tight_layout(); gcf()
end
